
# 1. Simple Kalman Filter

Consider a simple robotic example, where we can estimate the position and velocity of a mobile robot equiped with a sensor capable of measuring only its position (e.g. GPS like signal)

The robot motion is constrained to the 2D plane. Therefore, we consider the state of the robot $$ {\bf x}_{k} = [ x, y, \dot{x}, \dot{y} ] $$ contains its position ($x, y$)  and velocity ($\dot{x}, \dot{y}$) with respect to time.

The robot is observing (mesuring) its position acording to the following observation model:
$$ {\bf y}_{k} = C\:{\bf x}_{k} $$

$$ C = [[1, 0, 0, 0], [0, 1, 0, 0]] $$

Your task is to finish implementing the Kalman Filter class to track the position of the robot over time.

### **Simulation Setup**

Now, the code sets up a simulation with the Kalman filter tracking a robot over time.

```python
dt = 0.1
time_steps = 100
process_noise = 3.0
measurement_noise = 0.01
```
- **`dt`**: Time step of 0.1 seconds.
- **`time_steps`**: The number of steps to simulate.
- **`process_noise`** and **`measurement_noise`**: The noise in the process and the measurements.

```python
true_state = np.array([0, 0, 1, 1])
kf = KalmanFilter(dt, process_noise, measurement_noise, initial_state=true_state.copy())
```
- **`true_state`**: The true initial state of the robot (starting at position `(0, 0)` with velocity `(1, 1)`).
- **`kf`**: An instance of the Kalman filter.

### **Control Input and Sensor Measurement Simulation**

In the simulation setup we have provided two functions, __generate_u__ and __generate_y__. Before implementing the Kalman Filter in the next step you will need to implement these functions.

Both functions should return a tuple of size two, containing the generated value and the _noisy_ value.

- **`generate_u`** generates the control inputs for the robot. These could represent commands for the robot, like velocity. The function also adds noise to simulate real-world uncertainties. Initially, the simplest control could be command is a constant acceleration, but feel free to use any control command

- **`generate_y`** generates actual and noisy measurements from the robot's true state. The noise simulates errors in sensor readings and should be implemented by you.

After implementing both functions, the main simulation loop should be able to run correctly. Comment out any Kalman Filter related functions and run the loop, generating a trajectory plot using the code provided.

Ensure that the trajectory looks correct, given your implementation of the __generate_u__ function.

### **Kalman Filter Class**

The `KalmanFilter` class is a model that tracks the state of a system (in this case, the position and velocity of the robot) over time, based on noisy measurements and control inputs.

```python
class KalmanFilter:
    def __init__(self, dt, process_noise, measurement_noise, initial_state):
```
- The constructor (`__init__`) initializes the Kalman filter parameters.
  - `dt`: Time step, how often we update the position.
  - `process_noise`: The noise in the system's movement (how unpredictable the motion model is is).
  - `measurement_noise`: The noise in the sensor measurements (how accurate the sensor is).
  - `initial_state`:  ${\bf x}_{0}$ ie. the initial position and velocity of the system.

```python
        self.A = np.array([[1, 0, dt, 0],
                           [0, 1, 0, dt],
                           [0, 0, 1, 0],
                           [0, 0, 0, 1]])
```
- `A` is the **state transition matrix**. It describes how the state (position and velocity) evolves over time. The robot's position and velocity should be updated by this matrix in each step.
  - The 4x4 matrix is set up so that the position updates by adding velocity times the time step (`dt`).

```python
        self.B = np.array([[0, 0],
                           [0, 0],
                           [1, 0],
                           [0, 1]])
```
- `B` is the **control matrix**, which determines how the control inputs (`u`) affect the state. The inputs might be velocity commands or accelerations that change the state.

```python
        self.H = np.array([[1, 0, 0, 0],
                           [0, 1, 0, 0]])
```
- `H` is the **measurement matrix**, which defines how the sensor measurements are related to the state.

- `Q` is the **process noise covariance**, which models how unpredictable the system's movement is (i.e., how much we trust the system model).
- `R` is the **measurement noise covariance**, representing the uncertainty in the sensor measurements.
In your case you will need to construct `H`, `Q` and `R` appropiately

```python
        self.P = np.eye(4) * 1.0
        self.x = initial_state
```
- `P` is the **estimate covariance matrix**, which describes how confident we are in the current estimate of the state.
- `x` is the **initial state vector** that contains the initial position and velocity of the robot.

### **Kalman Filter Methods**
You need to implement the __predict__ and __update__ methods of the Kalman Filter.

At each intermediate step (ie, after predicting but before implementing the update) you should plot your estimated trajectory (some of the provided code will need to be modified) and _compare_ it with the ground truth trajectory.

After implementing the __predict__ function:
- Plot the covariance matrix (code provided) of the robots position. Without any measurements to correct its poisition, the covariance ellipse should grow over time.
- Experiment with different _control inputs_ and _process noise_ values. How does this effect the prediction of your robot state? Consider how the covariance of the robot state should change with different process noise values.

After implementing the __update__ function:
- Compare your prediction only trajectory with the trajectory when measurement updates are applied. Hopefully, the latter should be more accurate - how can you quantify this accuracy?
- Experiment with different _measurement noise_ values and see how this effects the resulting estimation.
- Experiment with updating the Kalman Filter with new measurements at different rates, simulating a sensor with a slow update rate. What happens to the estimated mean and covariance? When a measurement update occurs what should happen to the estimated covariance?

In both functions `self.x` and `self.P` will need to updated to reflect the new state mean and covariance.







In [1]:
import numpy as np
import matplotlib.pyplot as plt

from matplotlib.axes import Axes

def plot_position_covariance(axes: Axes, position: np.ndarray, position_covariance: np.ndarray):
  from matplotlib import patches
  w, v = np.linalg.eigh(position_covariance)
  k = 0.5
  x, y = position

  angle = np.arctan2(v[1, 0], v[0, 0])
  e1 = patches.Ellipse((x, y),
                          np.sqrt(w[0]),
                          np.sqrt(w[1]),
                          angle=np.rad2deg(angle),
                          fill=False,
                          color='orange')
  axes.add_artist(e1)
  return e1



class KalmanFilter:
    def __init__(self, dt, process_noise, measurement_noise, initial_state):
        self.dt = dt
        self.A = np.array([[1, 0, dt, 0],
                           [0, 1, 0, dt],
                           [0, 0, 1, 0],
                           [0, 0, 0, 1]])
        self.B = np.array([[0, 0],
                           [0, 0],
                           [1, 0],
                           [0, 1]])
        self.H = np.array([[1, 0, 0, 0],
                           [0, 1, 0, 0]])
        self.Q = process_noise * np.eye(4)
        self.R = measurement_noise * np.eye(2)
        self.P = np.eye(4) * 0.1
        self.x = initial_state

    def predict(self, u):
        self.x = self.A @ self.x + self.B @ u
        self.P = self.A @ self.P @ self.A.T + self.Q
        return self.x

    def update(self, z):
        y = z - self.H @ self.x
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        self.x = self.x + K @ y
        self.P = (np.eye(4) - K @ self.H) @ self.P
        return self.x

# Simulation parameters
dt = 0.1
time_steps = 100
process_noise = 0.1
measurement_noise = 0.1

# Initial state [x, y, vx, vy]
true_state = np.array([0, 0, 1, 1])
kf = KalmanFilter(dt, process_noise, measurement_noise, initial_state=true_state.copy())

# Ground truth trajectory
true_trajectory = []
predicted_trajectory = []
estimated_position_covariance = []
measured_positions = []

def generate_u(process_noise: float):
    u = np.array([0.1,-0.3])
    u_noisy = u + np.random.normal(0, np.sqrt(process_noise), 2)
    return u, u_noisy

def generate_y(x_true: np.ndarray, measurement_noise: float):
    y = x_true[:2]
    y_noisy = y + np.random.normal(0, np.sqrt(measurement_noise), 2)
    return y, y_noisy

time = np.arange(time_steps) * dt

np.random.seed(42)
for t in range(time_steps):
    # control input
    u, u_noisy = generate_u(process_noise)
    # simulation prediction
    true_state = kf.A @ true_state + kf.B @ u

    # measurement of position
    y, y_noisy = generate_y(true_state, measurement_noise)

    predicted_state = kf.predict(u_noisy)

    # # update at slower intervals to simulate slow sensor measurement
    if t % 5 ==0:
        kf.update(y_noisy)
    measured_positions.append(y_noisy)

    estimated_position_covariance.append(kf.P[:2, :2])
    true_trajectory.append(true_state[:2])
    predicted_trajectory.append(predicted_state[:2])

true_trajectory = np.array(true_trajectory)
predicted_trajectory = np.array(predicted_trajectory)
measured_positions = np.array(measured_positions)
estimated_position_covariance = np.array(estimated_position_covariance)


fig,ax = plt.subplots(figsize=(10, 6))

for cov, pos in zip(estimated_position_covariance, predicted_trajectory):
    plot_position_covariance(ax, pos, cov)

# Plot results
ax.plot(true_trajectory[:, 0], true_trajectory[:, 1], 'g-', label='Ground Truth')
ax.plot(measured_positions[:, 0], measured_positions[:, 1], 'rx', label='Measurements (Noisy)')
ax.plot(predicted_trajectory[:, 0], predicted_trajectory[:, 1], 'b-', label='Kalman Filter Estimate')
ax.legend()
ax.set_xlabel('X Position')
ax.set_ylabel('Y Position')
ax.set_title('Kalman Filter Tracking a Mobile Robot with Control Commands')

plt.show()